# System Identification <br><span style="font-size: 18px;">(by Nicolás García nicolas.garcia@tum.de)</span>

System Identification (SI) is a powerful set of tools used to create reduced-order models (ROM) of dynamic systems, typically from experimental data. The field of SI is an active area of research and the subject of numerous textbooks. A comprehensive text,  `System Identification: Theory for the User` was written by Ljung,  who also maintains the System Identification Toolbox in MATLAB, which serves as the foundation for much of the content in this work.

The goal of the present document is to provide a tutorial on the the CFD/SI methodology, which uses CFD simulations to generate time series data of thermoacoustic quantities, and SI to build the ROM from the generated data. Despite the very mature state of the Matlab SI toolbox, the free and open-source nature of the python ecosystem was chosen for this document, as well as the availability of the interactive jupyter notebook environment.

While SI can be used to build the reduced order models (ROMs) for a variety of systems, the use case in thermoacoustics will typically be the study of a turbulent combustor. In such a system, the relation between acoustic velocity fluctuations $u'$ as the input, and heat release fluctuations $q'$ as the output are the quantities of interest. The identified ROM can then be used to investigate the acoustic stability of the system without doing experiments or doing further computationally expensive LES. A very brief overview of the CFD/SI methodology will be given in this tutorial, based on the paper by [Polifke, 2014](https://www.sciencedirect.com/science/article/pii/S0306454913005768).

## Linear Time Invariant Systems

System identification of linear time invariant (LTI) systems are considered in this tutorial.  Let $\textbf{s}$ be the input signal and $\textbf{r}$ its corresponding response:

$$
\textbf{s} \rightarrow \textbf{r}
$$

For such a system to be linear, the principle of superposition must hold. This means that for two arbitrary input sequences $\textbf{s}^{(1)}$ and $\textbf{s}^{(2)}$, and their corresponding responces $\textbf{r}^{(1)}$ and $\textbf{r}^{(2)}$, then for arbitrary constants $a$ and $b$:
$$
a \textbf{s}^{(1)} + b \textbf{s}^{(2)} = a \textbf{r}^{(1)} + b \textbf{r}^{(2)}
$$

The time invariant condition just states that the system does not behave differently depending on the time of observation.

## Finite Impulse Response

Further restricting the scope of this tutorial, in addition to LTI systems, only single-input single-output (SISO) systems will be discussed. For LTI SISO systems, the Finite Impulse Response (FIR) is the simplest model available. The FIR of a system is conceptualized as a series of coefficients that constitute its response to a unit impulse, it can be written as:
$$
\textbf{r}[i] = \sum_{k=0}^{N-1} h[k] \textbf{s}[i-k]
$$

The sequence of coefficients $h[k]$ of length $N$ is the FIR of the system and is calculated using correlation analysis. The autocorrelation matrix for a signal of length $L$, using $N$ time lags is built in the following way:
$$
\Gamma_{i,j} = \frac{1}{L-N+1} \sum_{n=N}^{L} \textbf{s}[n-i] \textbf{s}[n-j]
$$
The cross correlation vector for the signals $\textbf{r}$ and $\textbf{s}$ using $N$ time lags is:
$$
c_i = \frac{1}{L-N+1} \sum_{n=N}^{L} \textbf{s}[n-i] \textbf{r}[n]
$$

Then, the impulse response is determined with the following linear system, called the Wiener-Hopf equation:

$$
\Gamma h = c
$$
Inversion of this linear system determines the FIR. The necessary length $N$ of the FIR is determined on a case by case basis.

## Data Acquisition.

Acquiring timeseries of the heat release rate and velocity fluctuations is the 'CFD' part of CFD/SI. Different CFD solvers like Fluent or OpenFOAM can be used for such a purpose. The recommended method for acquiring data is broadband excitation of a turbulent reacting flow. A signal which spans the complete spectrum of interest is developed, and subsequently used to perturb a CFD simulation which has reached a statistically steady state in its unperturbed form. The quantities of interest shall be extracted from the simulation by using the area weighted average of the velocity at the plane position and volume integrated heat release rate. Sufficiently long time series need to be generated, cutting the initial portion of the time series due to it being a transient stage.

## Tutorial

This tutorial contains time series from a laminar CFD simulation using compressible and incompressible formulations. The case that was simulated is a laminar slit flame, originally presented by [Kornilov et al., 2009](https://www.sciencedirect.com/science/article/pii/S0010218009002016). It is a multi-slit, laminar Bunsen flame, especifically designed to investigate its acoustic response. The following is a sketch of the experimental setup:

<center><img src="images/Kornilov-diagram.png" width="15%"></center>

The computational domain that was simulated to generate the data in this tutorial is the following:

<center><img src="images/Kornilov-cfd-diagram.png" width="10%" /></center>

A laminar case like this one provides a simple and cost-effective benchmark for validating and testing our methods. However, it is worth noting that SI can also handle more complex systems, including turbulent flames.

In [ ]:
import numpy as np
import scipy.signal as ss
import matplotlib.pyplot as plt
import time
from impulseest import impulseest
import pandas as pd

### SysId Settings

Ideally, the time series obtained from CFD should be as long as possible to ensure the robustness of the impulse response coefficient calculations. However, in practice, acquiring CFD data is expensive. Therefore, a time series length of approximately 10-20 times the impulse response duration is typically chosen: for an impulse response length of 10 ms, this corresponds to a time series of 100 to 200 ms. Longer time series may not be necessary if the noise intensity in the system (e.g., combustion noise in turbulent flames) is low.

The time interval between the coefficients of the impulse response corresponds to the sampling time of the signals used in the Wiener-Hopf inversion (see the equation above). If the time series is not downsampled, this can result in impulse responses with hundreds or thousands of coefficients, leading to an ill-conditioned system and a spurious solution. Therefore, the time series must be downsampled to ensure that the number of impulse response coefficients—our quantities of interest in the system identification problem—remains between 10 and 100. In the second step, the initial transient of the time series is removed from the input and output signals. This ensures that the system under evaluation is statistically steady and well represented by an FIR model. For this particular case, the following values are chosen.

In [ ]:
IRlength = 10e-3 #impulse response length is case dependent
dtResample = 1e-4 # resampling time step for system identification. 

tStart = 0.032 # where to cut the signal at the start (e.g. cutting transients in the simulation)?
tEnd = 0.232 # where to cut the signal at the end (e.g. to investigate sensitivity to total time series length)?
nResampled = int((tEnd-tStart)/dtResample)

freq_max = 1000 # maximum frequency

### Input - Output Signals

The forcing signal can be constructed using a low-pass filtered white noise, which can be associated with a Discrete Random Binary Signal (DRBS) to enhance the crest factor. Note that the input signal does not necessarily correspond to the signal used to force the system. In compressible CFD, the signal used to force the system is acoustic in nature, whereas the input signal corresponds to the time series of the quantity of interest (e.g., velocity) at a given reference position. In incompressible CFD, the forcing signal generally corresponds to velocity fluctuations and may only differ slightly from the input signal if the location of the forcing is close to that of the reference location. [Eder et al., 2024](https://journals.sagepub.com/doi/full/10.1177/17568277231154204) 

In [ ]:
#casename='data_u5ampl_CM2_comp/'
#casename='data_u5ampl_CM2_incomp/'
casename='isothermalKornilov/'
U = pd.read_csv(casename+'velocityRef.csv',names=['t','U'])
Q = pd.read_csv(casename+'heatRelease.csv',names=['t','Q'])
t, u= np.split(U.to_numpy(),2,axis=1)
t, q= np.split(Q.to_numpy(),2,axis=1)
dt = np.mean(t[1:]-t[0:-1])

In [ ]:
# Cut time series to be the same length

idStart = np.searchsorted(np.squeeze(t),tStart)
idEnd = np.searchsorted(np.squeeze(t),tEnd)

t = t[idStart:idEnd] - tStart
u = u[idStart:idEnd]
q = q[idStart:idEnd]

# Normalize and detrend

u_norm = (u - np.mean(u))/np.mean(u)
q_norm = (q - np.mean(q))/np.mean(q)

# Resample arrays
u_ds, t_ds = ss.resample(u_norm,nResampled,t)
q_ds = ss.resample(q_norm, nResampled)
dtResampled = np.mean(t_ds[1:]-t_ds[0:-1])

### Time Domain Plot

In [ ]:
fig, axs = plt.subplots(2, 1, layout='constrained', sharex=True,figsize=(12,6),dpi=200)
axs[0].plot(t, u)
axs[0].set_ylabel('U')
axs[0].grid(True)
axs[0].set_title('velocityRef')

axs[1].plot(t, q)
axs[1].set_xlim(0, t[-1])
axs[1].set_xlabel('Time (s)')
axs[1].set_ylabel('Q')
axs[1].grid(True)
axs[1].set_title('heatRelease')

fig.suptitle('Input Output Data', fontsize=16)

plt.show()

### Autocorrelation

The autocorrelation of the signal is plotted to investigate its statistical independence. A very uncorrelated signal will yield a very well conditioned system, suited for precise identification. Ideally, the signal would show 0 autocorrelation after the first time-lag, but it is acceptable for the correlation to approach zero after just a few steps.

In [ ]:
fig,ax = plt.subplots(figsize=(12,4),dpi=300)
ax.acorr(np.squeeze(u_ds), usevlines=True, normed=True, maxlags=50, lw=2)
ax.set_xlim(0,50)
ax.grid(True)
ax.set_ylabel("$\gamma$  [-]",fontsize=11)
ax.set_xlabel("Time lags [-]",fontsize=11)
ax.set_title('Auto-correlation of Input Signal')
plt.show()

### Impulse Response

Calculation of the impulse response of the system is done using the `impulseest` function. Different regularization kernels are tested. The need for a regularization kernel is discussed by [Vinícius et al, 2021](https://www.sciencedirect.com/science/article/pii/S2352711021000832), which are the authors of the `impulseest` package.

In [ ]:
N_FIR = int(np.ceil(IRlength/dtResample))

reg = 'none'
start_time = time.time()
ir_none = impulseest(u_ds,q_ds,n=N_FIR,RegularizationKernel=reg)
end_time = time.time()
reg1 = 'TC'
start_time1 = time.time()
ir_TC = impulseest(u_ds,q_ds,n=N_FIR,RegularizationKernel=reg1)
end_time1 = time.time()
reg2 = 'DC'
start_time2 = time.time()
ir_DC = impulseest(u_ds,q_ds,n=N_FIR,RegularizationKernel=reg2)
end_time2 = time.time()

In [ ]:
print(f'No kernel was used for regularization, and it took {(end_time-start_time):.2f} seconds.')
print(f'Kernel {reg1} was used for regularization, and it took {(end_time1-start_time1):.2f} seconds.')
print(f'Kernel {reg2} was used for regularization, and it took {(end_time2-start_time2):.2f} seconds.')

### Plotting

In [ ]:
#plotting and saving impulse responses
fig,ax1 = plt.subplots(figsize=(12,4),dpi=300)
lns3 = ax1.plot(ir_TC,label="IR with TC regularization",linewidth=1.5,color='C2')
lns2 = ax1.plot(ir_DC,label="IR with DC regularization",linestyle='dashed',linewidth=1.5,color='C1')
lns1 = ax1.plot(ir_none, label="IR with No regularization",linestyle='dotted',linewidth=1.5,color='C0')
ax1.set_xlabel("Samples [p.u.]",fontsize=11)
ax1.set_ylabel("Impulse response value [p.u.]",fontsize=11)
ax1.set_xlim([0,100])

lns = lns1+lns2+lns3
labs = [l.get_label() for l in lns]
ax1.legend(lns,labs,loc='lower right')
ax1.grid()

plt.show()

In [ ]:
#plotting and saving impulse responses
fig,ax1 = plt.subplots(figsize=(12,4),dpi=300)
lns3 = ax1.plot(ir_TC,label="IR with TC regularization",linewidth=1.5,color='C2')
lns2 = ax1.plot(ir_DC,label="IR with DC regularization",linestyle='dashed',linewidth=1.5,color='C1')
ax1.set_xlabel("Samples [p.u.]",fontsize=11)
ax1.set_ylabel("Impulse response value [p.u.]",fontsize=11)
ax1.set_xlim([0,100])

lns = lns2+lns3
labs = [l.get_label() for l in lns]
ax1.legend(lns,labs,loc='lower right')
ax1.grid()

plt.show()

### Frequency Domain, and the Flame Transfer Function

The FIR corresponds to a time domain response of the system. The frequency domain response is the Flame Transfer Function (FTF), which is related to the impulse response by means of Z transform:

$$
\mathcal{F}(\omega) = \sum_{k=0}^N h_k e^{i \omega k \Delta t}
$$

In [ ]:
ir_arr = np.squeeze(np.asarray(ir_TC))
ids = np.arange(0,len(ir_arr))
freqvec = np.linspace(0,freq_max,100)

ftf = np.zeros_like(freqvec,dtype=complex)
for i,freq in enumerate(freqvec):
    z = np.exp(-1j*2*np.pi*freq*dtResampled)
    z_vec = z**ids
    ftf[i]=np.sum(ir_arr*z_vec)

In [ ]:
fig, axs = plt.subplots(2, 1,figsize=(6,8), layout='constrained', sharex=True)
axs[0].plot(freqvec, np.abs(ftf), label='SI')
if(casename=='isothermalKornilov/'):
    gain = pd.read_csv('isothermalKornilov/gain.csv')
    axs[0].scatter(gain['freq'], gain['gain'], label='Experiment',color='C2')
    axs[0].legend()
axs[0].set_ylabel('Gain (abs)')
axs[0].set_ylim(0.01,2)
axs[0].grid(visible=True,which='both')
#axs[0].set_title('from: velocityRef to:heatRelease')

axs[1].plot(freqvec, np.unwrap(np.angle(ftf)),label='SI')
if(casename=='isothermalKornilov/'):
    phase = pd.read_csv('isothermalKornilov/phase.csv')
    axs[1].scatter(phase['freq'], -1*phase['phase'], label='Experiment',color='C2')
    axs[1].legend()
axs[1].set_xlim(0, 600)
axs[1].set_xlabel('Frequency (Hz)')
axs[1].set_ylabel('Phase (rad)')
axs[1].grid(True)

fig.suptitle('Bode Diagram', fontsize=16)

plt.show()

### Time Domain, using the Impulse Response

The impulse response can be used to predict the output signal in time domain directly, using its convolution with the input signal. Below, a comparison of the original output signal and the signal computed from the convolution of the input with the impulse response is shown.

In [ ]:
output_from_ir = ss.convolve(np.squeeze(u_ds),ir_arr, mode='full')
output_from_ir=output_from_ir[0:t_ds.size]
fig, axs = plt.subplots(2, 1, layout='constrained', sharex=True,figsize=(12,6),dpi=200)
axs[0].plot(t_ds,q_ds)
axs[0].set_title('Original Output Signal')
axs[0].set_xlim(0, 0.5)
axs[1].plot(t_ds,output_from_ir)
axs[1].set_title('Output from Impulse Response')
axs[1].set_xlim(0, t_ds[-1])
plt.show()

In [ ]:
fig,ax1 = plt.subplots(figsize=(12,4),dpi=300)
lns3 = ax1.plot(t_ds,q_ds,label="Output from IR",linewidth=1.5,color='C2')
lns2 = ax1.plot(t_ds,output_from_ir,label="Original output signal",linestyle='dashed',linewidth=1.5,color='C1')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Q')
ax1.grid(True)
ax1.set_xlim([0,t_ds[-1]])

lns = lns2+lns3
labs = [l.get_label() for l in lns]
ax1.legend(lns,labs,loc='lower right')
ax1.grid()
fig.suptitle('Time Domain Plot, Fitted vs Original Data', fontsize=16)

plt.show()

As this plot shows, a very good prediction can be made on the output signal with the constructed model. This verifies that SI is a very powerful and approachable tool to identify flame transfer functions and impulse responses!

### Next steps

With the knowledge of identifying impulse responses and their relation to the frequency response, it is possible to investigate more complex cases like turbulent flames, or doing parameter variations on the same case to perform sensitivity analysis. Doing MISO (Multiple Input Single Output) is also possible, for studying equivalence ratio variations. Plenty of possibilities are available to extend and apply the procedure presented here, so go ahead an try them!